# Notebook 2: Inspect, Iterate, Train

**Phases 3–4** | ~35 minutes

In this notebook you will:
1. Visualize and inspect your auto-generated labels
2. Run error analysis — find and fix label quality issues
3. Filter noisy/tiny labels to improve training signal
4. Train a YOLO26n student model from the curated labels
5. Monitor training curves

> **Key insight you'll discover**: Label quality matters more than data quantity.
> A saturating curve (R²=0.97) shows that more images alone cannot reach production quality.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

WORKSHOP_DIR = Path(".").resolve().parent
DATA_DIR = WORKSHOP_DIR / "data"

# Set this to YOUR labeled dataset from Notebook 1
LABELED_DIR = DATA_DIR / "ppe_dataset_sam3_exp_a"  # adjust if you used a different name
print(f"Dataset: {LABELED_DIR}")
print(f"Exists: {LABELED_DIR.exists()}")

---
## 1. Phase 3 — Error Analysis

### Before you train: LOOK at the labels.

Auto-labelers aren't perfect. Common issues:
- **Tiny labels** (<20px) that the model can't learn from
- **Spurious detections** on background objects
- **Missing detections** on occluded workers
- **Loose bounding boxes** that include too much background

### Tools for Error Analysis

```bash
# Visualize labels overlaid on images
python data/visualize_gt_annotations.py \
    --dataset-dir data/ppe_dataset_sam3_exp_a \
    --output-dir data/label_viz \
    --split val

# Filter out tiny/noisy labels
python data/filter_tiny_labels.py \
    --input-dir data/ppe_dataset_sam3_exp_a \
    --output-dir data/ppe_dataset_sam3_exp_a_filtered \
    --min-dim 0.03
```

In [ ]:
# ── Utility: Side-by-side before/after comparison ────────────────────────────

def compare_before_after(image_path, label_before, label_after, class_names=None, figsize=(18, 7)):
    """Show an image with labels before and after filtering, side by side."""
    img = np.array(Image.open(image_path))
    h, w = img.shape[:2]
    
    CLASS_COLORS = {
        0: (0, 200, 0),      # hardhat — green
        1: (220, 30, 30),     # person — red
        2: (30, 100, 220),    # blue
    }
    
    def draw_labels(ax, label_path, title):
        ax.imshow(img)
        count = 0
        if Path(label_path).exists():
            for line in Path(label_path).read_text().strip().splitlines():
                parts = line.split()
                cls_id = int(parts[0])
                cx, cy, bw, bh = [float(x) for x in parts[1:5]]
                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w)
                y2 = int((cy + bh/2) * h)
                
                color = np.array(CLASS_COLORS.get(cls_id, (200, 200, 200))) / 255
                name = class_names.get(cls_id, f"cls_{cls_id}") if class_names else f"cls_{cls_id}"
                rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                     linewidth=2, edgecolor=color, facecolor="none")
                ax.add_patch(rect)
                ax.text(x1, y1-4, name, color=color, fontsize=7, fontweight="bold",
                        bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.7))
                count += 1
        ax.set_title(f"{title} ({count} labels)", fontsize=11)
        ax.axis("off")
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    draw_labels(ax1, label_before, "Before Filtering")
    draw_labels(ax2, label_after, "After Filtering")
    plt.tight_layout()
    plt.show()

In [ ]:
# TODO: Run error analysis
# Step 1: Visualize labels
#   !python {DATA_DIR}/visualize_gt_annotations.py \
#       --dataset-dir {LABELED_DIR} \
#       --output-dir {DATA_DIR}/label_viz \
#       --split val
#
# Step 2: Look at the visualizations — what do you notice?
#
# Ask Claude Code: "Visualize my labels and help me identify quality issues"


### Tiny Label Filtering

**Key insight**: Labels smaller than ~3% of the image dimension (~20px at 640px)
are too small for the model to learn from. They add noise to training.

In our experiments, filtering tiny labels:
- Removed **35.6%** of all labels (noise!)
- Raised mAP50 by **+2.7%**
- Improved person recall from 71.6% to **89.8%**

In [ ]:
# TODO: Filter tiny labels
#   !python {DATA_DIR}/filter_tiny_labels.py \
#       --input-dir {LABELED_DIR} \
#       --output-dir {LABELED_DIR}_filtered \
#       --min-dim 0.03
#
# Ask Claude Code: "Filter tiny labels from my dataset"


In [ ]:
# TODO: Compare before and after filtering on a few images
# Use the compare_before_after() function defined above
#
# Example:
#   FILTERED_DIR = LABELED_DIR.parent / f"{LABELED_DIR.name}_filtered"
#   val_images = sorted((LABELED_DIR / "images" / "val").glob("*.*"))
#   for img_path in val_images[:3]:
#       stem = img_path.stem
#       compare_before_after(
#           img_path,
#           LABELED_DIR / "labels" / "val" / f"{stem}.txt",
#           FILTERED_DIR / "labels" / "val" / f"{stem}.txt",
#           class_names={0: "hardhat", 1: "person"},
#       )


### Phase 3 Observations

- Labels removed by filtering: ___%
- Most common issue: ___
- Were there any false positives? ___
- Which scene categories had the worst labels? ___

---
## 2. Phase 4 — Train YOLO26n

Now we train a fast, specialized student model using our curated labels.

### Training Configuration

| Parameter | Value | Why |
|-----------|-------|-----|
| Model | `yolo26n.pt` | Nano variant — fast inference, small footprint |
| Epochs | 100 | With patience=20 early stopping |
| Batch | 8 | Safe for A40 GPU memory |
| Image size | 640 | Standard YOLO training resolution |
| Workers | 0 | Avoids multiprocessing issues on HPC |

### Option A: Use the training script

```bash
# Edit train_baseline_ppe.py to point DATA at your filtered dataset's dataset.yaml
python data/train_baseline_ppe.py
```

### Option B: Train inline (more flexible)

```python
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
results = model.train(
    data="path/to/your/dataset.yaml",
    epochs=100,
    patience=20,
    batch=8,
    imgsz=640,
    device="cuda",
    workers=0,
    project="data/ppe_results",
    name="my_experiment",
)
```

> **Short on time?** Pre-baked results are available at `data/ppe_results/`

In [ ]:
# TODO: Train YOLO26n
# Use either the script or the inline approach.
#
# Ask Claude Code: "Train a YOLO26n model on my filtered dataset"


In [ ]:
# ── Utility: Plot training curves ────────────────────────────────────────────

import pandas as pd

def plot_training_curves(results_csv, figsize=(16, 5)):
    """Plot training curves from Ultralytics results.csv."""
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()  # clean whitespace from column names
    
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    # Loss curves
    loss_cols = [c for c in df.columns if "loss" in c.lower() and "val" not in c.lower()]
    val_loss_cols = [c for c in df.columns if "loss" in c.lower() and "val" in c.lower()]
    
    ax = axes[0]
    for col in loss_cols:
        ax.plot(df["epoch"], df[col], label=col, alpha=0.8)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    
    # Validation loss
    ax = axes[1]
    for col in val_loss_cols:
        ax.plot(df["epoch"], df[col], label=col, alpha=0.8)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Validation Loss")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    
    # mAP curves
    ax = axes[2]
    map_cols = [c for c in df.columns if "map" in c.lower() or "mAP" in c]
    for col in map_cols:
        ax.plot(df["epoch"], df[col], label=col, alpha=0.8, linewidth=2)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("mAP")
    ax.set_title("Validation mAP")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print best metrics
    if map_cols:
        best_col = [c for c in map_cols if "50" in c and "95" not in c]
        if best_col:
            best_epoch = df[best_col[0]].idxmax()
            best_val = df[best_col[0]].max()
            print(f"\nBest mAP50: {best_val:.4f} at epoch {int(df['epoch'].iloc[best_epoch])}")

In [ ]:
# TODO: Plot training curves
# Find the results.csv in your training output directory
#
# Example:
#   RESULTS_DIR = DATA_DIR / "ppe_results" / "my_experiment"
#   plot_training_curves(RESULTS_DIR / "results.csv")
#
# Or for pre-baked results:
#   plot_training_curves(DATA_DIR / "ppe_results" / "baseline_construction_ppe" / "results.csv")


### Phase 4 Observations

- Training time: ___ minutes
- Best mAP50: ___
- Best epoch: ___ (out of how many?)
- Did early stopping trigger? ___

---

**Next**: Open `03_evaluate_and_deploy.ipynb` to evaluate the model and build a compliance system.